# Data Cleaning

The goal of this notebook is to clean and validate the customer churn dataset
before exploratory analysis and business intelligence workflows.

In particular, we will:
- Check for duplicate customers
- Validate data consistency
- Handle missing values
- Verify numeric integrity
- Prepare the dataset for SQL analysis and dashboard development

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
df.head()

---
## Initial Inspection

We start by inspecting the dataset structure, including number of rows,
columns, and basic information.

In [ ]:
df.shape

In [ ]:
df.info()

The initial inspection indicates that the dataset contains no explicit null values. However, additional inspection is required since missing values may still be represented as empty strings.

---
## Missing Values

In [ ]:
missing_values = df.select_dtypes(include="object").apply(
    lambda col: (col.str.strip() == "") | col.isna()
).sum()
missing_values

Missing values were found in the TotalCharges column. To better understand these cases, we inspect the affected rows.

In [ ]:
inconsistency = df[(df["TotalCharges"].str.strip() == "" ) | (df["TotalCharges"].isna())]
inconsistency

All missing values are in rows with tenure = 0, which suggests that these customers are new and have not accumulated charges yet. So, we are going to fill the total charge with 0 to be consistent with the tenure information and resolve the missing value issue. Additionally, we change the column's data type to float.

In [ ]:
df["TotalCharges"] = df["TotalCharges"].str.strip().replace("", 0)
df["TotalCharges"] = df["TotalCharges"].astype(float)
df.info()

---
## Checking for Duplicate Users

Each customer should appear only once in the dataset. Therefore, we verify whether customer identifiers are unique.

In [ ]:
duplicate_summary = pd.Series({
    "duplicate_user_ids": df["customerID"].duplicated().sum(),
    "is_unique": df["customerID"].nunique() == len(df)
})

duplicate_summary

No duplicated customer records were identified.

---
## Numerical Consistency

### Negative Values
All numeric columns in this dataset are expected to contain only non-negative values. We then check if there is any negative value in them. 

In [ ]:
numeric_cols = ["MonthlyCharges", "TotalCharges", "tenure", "SeniorCitizen"]

(df[numeric_cols] < 0).sum()

No negative value was found in the numeric columns, indicating no apparent integrity issues in the numeric fields.

### Charge and Tenure Consistency
Since tenure represents the amount of time a customer has been using the service, it would not be consistent to have:
- a tenure equals 0 and some charge;
- a tenure greater than 0 and no charge.

In [ ]:
inconsistent_charges = df[
    ((df["tenure"] == 0) & (df["TotalCharges"] > 0)) | 
    ((df["tenure"] > 0) & (df["TotalCharges"] == 0))
]

inconsistent_charges

No records with these inconsistencies were found, indicating consistency between tenure and billing.

### Senior Citizen Values

Since SeniorCitizen is a binary indicator, the column must have only zeros and ones. To validate this assumption, we observe the unique values in the column.

In [ ]:
set(df["SeniorCitizen"].unique())

Only valid binary values were identified.

---
## Categorical Consistency

We check the values in the categorical columns to identify any potential inconsistencies in categorical values, as illogical types and typos.

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].unique())

No apparent inconsistencies or typographical issues were identified in the categorical variables.

---
## Final Dataset Overview

The dataset was successfully cleaned and validated, with the identified inconsistencies properly addressed.

The data is now ready for SQL analysis, exploratory analysis, and dashboard development.

### Saving Cleaned Dataset

In [ ]:
df.to_csv("../data/processed/customer_churn_clean.csv", index=False)